# Least Privilege, Blast Radius, Scoped Tokens

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 41 §41.5–41.7 · type: walkthrough*

You'll constrain *capability*: tier tools by consequence, sandbox code execution with locked-down egress, and mint **delegated, per-user, short-lived** tokens — including the awkward background-worker case — so an injected agent's blast radius is **one user, not the platform**.

> ⚠️ **Defensive framing (hard rule).** Everything here is *defender-side*. Every “attack” is a benign, clearly-labeled test fixture using obviously-fake payloads (`evil.example`) so we can exercise — and **measure** — our own defenses. Nothing targets a real system or model.

## 🧠 Why this matters

Guardrails inspect *content* (41-02); this layer constrains **capability** — and capability is where the real damage lives (OWASP **Excessive Agency**). The antidote is the oldest idea in security applied with fresh seriousness: **least privilege**. Give each agent the minimum tools, tier them by consequence, sandbox execution, and make tools act **as the requesting user** via scoped, short-lived tokens. Then a hijacked agent inherits *one user's* reach, not the platform's — which converts a platform-ending breach into a one-user incident.

## Objectives & prereqs

**By the end you can**
- route tool calls through a **consequence-tiered** approval policy;
- scope each tool's credential to the minimum, and **sandbox** code with a default-deny egress;
- bound blast radius (rate/spend/iteration caps, kill switch, audit log);
- implement **OAuth2 token exchange** (RFC 8693) and the **background-worker** delegation against a mock IdP.

**Prereqs:** 41-01–02; Ch 12 (tool safety), Ch 20 (approvals), Ch 26 (authn/z, OBO), Ch 31 (Celery workers), Ch 30 (per-tenant isolation). Runs **fully offline** — tools, IdP, and sandbox are simulated.

In [ ]:
# --- Setup -------------------------------------------------------------------
import os
import random
import secrets
import time
from dataclasses import dataclass, field
from enum import IntEnum

from dotenv import load_dotenv

load_dotenv()  # local .env if present; never hardcode keys

# MOCK=1 (default): tools, the IdP, the sandbox, and the agent are all SIMULATED and
# deterministic -- no cloud, no real tokens, no real container. Any 'side effect' is a
# dry-run log line. MOCK=0 is not required; there is nothing live to call.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

random.seed(41)

if MOCK:
    print("MOCK mode: simulated tools/IdP/sandbox. No cloud, no real tokens.")
else:
    print("NOTE: simulated by design; MOCK=0 changes nothing here.")

## 1 · Tool tiers by consequence

Build the book's table as a **policy map**, ordered by how reversible the action is — the same reversibility instinct from Ch 27, here setting the approval policy per tier.

| Tier | Examples | Policy |
|---|---|---|
| Read-only | search, fetch document, get status | Auto-approve |
| Reversible write | draft email, create ticket, add comment | Auto + audit log |
| Hard to reverse | send email, post publicly, modify records | Confirm or review |
| Irreversible / high value | delete data, move money, deploy, grant access | Human approval, always |

In [ ]:
class Tier(IntEnum):
    READ_ONLY = 0
    REVERSIBLE_WRITE = 1
    HARD_TO_REVERSE = 2
    IRREVERSIBLE = 3

POLICY = {
    Tier.READ_ONLY: "auto",
    Tier.REVERSIBLE_WRITE: "auto+audit",
    Tier.HARD_TO_REVERSE: "confirm",
    Tier.IRREVERSIBLE: "human_approval",
}

TOOL_TIERS = {
    "search": Tier.READ_ONLY,
    "read_ticket": Tier.READ_ONLY,
    "draft_reply": Tier.REVERSIBLE_WRITE,
    "send_email": Tier.HARD_TO_REVERSE,
    "delete_data": Tier.IRREVERSIBLE,
    "move_money": Tier.IRREVERSIBLE,
}

def requires_human(tool: str) -> bool:
    return POLICY[TOOL_TIERS[tool]] == "human_approval"

### 🔮 Predict

We'll route a batch of mock tool calls through the policy. **Which require a human click** (`human_approval`)? Predict the set before running.

In [ ]:
calls = ["search", "draft_reply", "send_email", "delete_data", "move_money"]
for tool in calls:
    tier = TOOL_TIERS[tool]
    print(f"{tool:<12} {tier.name:<18} -> {POLICY[tier]}")

needs_human = [t for t in calls if requires_human(t)]
print("\nhuman approval required:", needs_human)

**What you just saw.** Only the *irreversible* actions stop for a human — the reversibility tier *is* the approval policy. Read-only flows stay fast; the expensive-to-undo ones get a gate.

## 2 · Least-privilege permissions

A support agent gets `read_ticket` + `draft_reply`, **not** `execute_sql`. And scope the *credential behind* each tool to the same minimum (read-only on two tables, one API scope) — so the agent physically cannot read tenant B's data on behalf of tenant A.

In [ ]:
AGENT_TOOLS = {
    "support_agent": {"read_ticket", "draft_reply"},   # NOT execute_sql
    "billing_agent": {"read_ticket", "move_money"},
}

def can_use(agent: str, tool: str) -> bool:
    return tool in AGENT_TOOLS.get(agent, set())

for tool in ["read_ticket", "execute_sql", "move_money"]:
    print(f"support_agent -> {tool:<12}: {'allowed' if can_use('support_agent', tool) else 'DENIED'}")

## 3 · Sandboxing code execution

A code-interpreter tool runs in a **disposable container**: no ambient secrets, read-only filesystem, **egress allowlist (ideally nothing)**, CPU/mem/time limits, destroyed after the task. The point: **data that can't leave the sandbox can't be exfiltrated** — the last line, complementing 41-02's egress proxy. We *simulate* the container (no real container in CI).

In [ ]:
@dataclass
class Sandbox:
    egress_allowlist: frozenset = frozenset()  # ideally empty
    cpu_seconds: float = 2.0
    secrets_present: bool = False              # no ambient secrets
    readonly_fs: bool = True
    log: list = field(default_factory=list)

    def attempt_egress(self, host: str) -> bool:
        ok = host in self.egress_allowlist
        self.log.append(("egress", host, "ALLOW" if ok else "DENY"))
        return ok

    def attempt_write(self, path: str) -> bool:
        ok = not self.readonly_fs
        self.log.append(("write", path, "ALLOW" if ok else "DENY"))
        return ok

    def destroy(self) -> None:
        self.log.append(("destroy", "", "OK"))  # nothing persists past the task

# A code-interpreter task that has been injected to exfiltrate the (fake) secret.
sbx = Sandbox(egress_allowlist=frozenset())  # default-deny: nothing
leaked = sbx.attempt_egress("evil.example")
wrote = sbx.attempt_write("/etc/passwd")
sbx.destroy()
for entry in sbx.log:
    print(entry)
assert not leaked and not wrote, "sandbox must contain the task"

**What you just saw.** Even a fully-injected interpreter task can't reach `evil.example` or write the filesystem — the container, not the prompt, drew the boundary, and it's gone after the task.

## 4 · Blast radius: caps, kill switch, audit log

Bound the damage assuming the layers fail: **rate/spend caps** per agent and per tenant (LLM10 / Ch 40), **iteration caps** on loops, **per-tenant isolation** (Ch 30), a **kill switch** that pauses an agent fleet in one action, and an **immutable audit log** of every tool call with arguments — your forensics.

In [ ]:
AUDIT: list = []  # append-only; keyed by tenant, user, agent, tool, args
FLEET_PAUSED = {"value": False}  # the kill switch

@dataclass
class Limits:
    max_calls: int = 3
    used: int = 0
    def check(self) -> bool:
        self.used += 1
        return self.used <= self.max_calls

def kill_switch(on: bool) -> None:
    FLEET_PAUSED["value"] = on

def invoke_tool(tenant, user, agent, tool, args, limits) -> str:
    if FLEET_PAUSED["value"]:
        return "PAUSED (kill switch)"
    if not can_use(agent, tool):
        return f"DENIED: {agent} lacks {tool}"
    if not limits.check():
        return "RATE-LIMITED (iteration/spend cap)"
    AUDIT.append({"tenant": tenant, "user": user, "agent": agent, "tool": tool, "args": args})
    return f"ok: {tool}"

lim = Limits(max_calls=2)
print(invoke_tool("acme", "alice", "support_agent", "read_ticket", {"id": 7}, lim))
print(invoke_tool("acme", "alice", "support_agent", "read_ticket", {"id": 8}, lim))
print(invoke_tool("acme", "alice", "support_agent", "read_ticket", {"id": 9}, lim))  # capped
kill_switch(True)
print(invoke_tool("acme", "alice", "support_agent", "read_ticket", {"id": 10}, lim))  # paused
kill_switch(False)
print("\naudit rows:", len(AUDIT))

## 5 🔧 Delegated authorization: OAuth2 token exchange (RFC 8693)

The wrong fix is a fat service account. The right one: exchange the *user's* token for a **new, narrowly-scoped, short-lived** one (`read:documents`, minutes) minted per run. A hijacked agent calling the tool 1000× is still **Alice-scoped** and still **expires**. We build the book's `exchange_token` / `redeem` shapes against a **mock IdP** — no real tokens, no network.

In [ ]:
@dataclass
class Grant:
    subject: str       # 'acting for Alice'
    audience: str      # which downstream API
    scope: tuple       # narrow, per-run
    expires_at: float  # short-lived
    opaque: str        # the redeemable handle

class MockIdP:
    """Simulated identity provider. Issues + redeems scoped, short-lived grants."""
    def __init__(self) -> None:
        self._issued: dict = {}

    def exchange_token(self, subject_token: str, audience: str,
                       scope: tuple, lifetime_seconds: int) -> Grant:
        subject = subject_token.removeprefix("user-token-for-")  # 'alice'
        handle = secrets.token_hex(8)  # opaque handle (not a real credential)
        g = Grant(subject, audience, tuple(scope), time.time() + lifetime_seconds, handle)
        self._issued[handle] = g
        return g

    def redeem(self, grant: Grant) -> Grant:
        live = self._issued.get(grant.opaque)
        if live is None or live.expires_at < time.time():
            raise PermissionError("grant expired or unknown")
        return live

idp = MockIdP()
grant = idp.exchange_token(
    subject_token="user-token-for-alice",  # proves 'this is Alice'
    audience="documents-api",
    scope=("read:documents",),              # narrow, per-run
    lifetime_seconds=600,                    # minutes, not months
)
print("minted grant:", grant.subject, grant.scope, "exp in ~", int(grant.expires_at - time.time()), "s")

In [ ]:
# A downstream API that enforces the grant: it serves ONLY the subject's data and
# ONLY within scope. A token minted for Alice can never read Bob's docs.
DOCS = {"alice": ["alice/report.md"], "bob": ["bob/secret.md"]}

def documents_api_list(grant: Grant, requested_user: str) -> list:
    if "read:documents" not in grant.scope:
        raise PermissionError("missing read:documents scope")
    if requested_user != grant.subject:
        raise PermissionError(f"403: token for {grant.subject} cannot read {requested_user}")
    return DOCS[requested_user]

print("alice reads alice:", documents_api_list(grant, "alice"))
try:
    documents_api_list(grant, "bob")  # injected agent tries cross-user
except PermissionError as e:
    print("alice reads bob ->", e)

## 6 · The background-worker case (the capstone hits it)

A Celery worker (Ch 31) wakes long after the HTTP request returned — **no live session** to borrow authority from. The wrong fix: a broad worker credential “to act for anyone.” The right fix: **carry the delegation into the job** — the web tier enqueues a scoped grant; the worker redeems it. The *job*, not the worker, carries the authority, so blast radius is **one user per job**.

In [ ]:
# Web tier: exchange the user's token, enqueue a delegated grant -- never a broad
# worker credential. (`.delay(...)` is simulated; no real Celery/broker here.)
def web_tier_enqueue(user_token: str, job_id: str) -> dict:
    grant = idp.exchange_token(
        subject_token=user_token,
        audience="documents-api",
        scope=("read:documents",),
        lifetime_seconds=600,
    )
    return {"job_id": job_id, "delegated_grant": grant}  # what .delay() would carry

# Worker: act STRICTLY within the grant it was handed.
def process_report(job_id: str, delegated_grant: Grant) -> list:
    token = idp.redeem(delegated_grant)          # scoped to one user, expiring
    return documents_api_list(token, token.subject)  # 403 for anyone but the subject

msg = web_tier_enqueue("user-token-for-alice", job_id="job-1")
print("worker output:", process_report(**msg))

### ⚠️ Pitfall: the fat worker credential

Giving the worker one broad credential “to act for anyone” turns a single poisoned job into **cross-tenant access**. Watch the difference:

In [ ]:
# WRONG: a fat worker credential that can read any user. One poisoned job -> everyone.
class FatServiceAccount:
    subject = "*"  # 'anyone' -- the bug
    def list_any(self, user: str) -> list:
        return DOCS[user]  # no per-user boundary at all

fat = FatServiceAccount()
print("fat worker reads bob too:", fat.list_any("bob"), "  <- platform-wide blast radius")

# RIGHT (above): the per-user grant 403s on cross-user reads. Blast radius = one user.
alice_grant = idp.exchange_token("user-token-for-alice", "documents-api",
                                 ("read:documents",), 600)
try:
    documents_api_list(alice_grant, "bob")
except PermissionError:
    print("scoped worker: cross-user read is impossible by construction")

## 7 · Per-tenant credential isolation (the structural backstop)

Delegation gets the *user* right; **tenant isolation guarantees the boundary** even if delegation has a bug. Separate signing **audiences** + row-level checks make it *physically impossible* for a tenant-A token to authorize a tenant-B action — **two independent controls**, so one mistake in either can't cross a tenant line.

In [ ]:
def authorize(token_tenant, resource_tenant, token_audience, resource_audience) -> bool:
    # BOTH must match: audience (signing boundary) AND tenant (row-level). Two controls.
    return token_tenant == resource_tenant and token_audience == resource_audience

print("acme->acme, same aud :", authorize("acme", "acme", "acme-aud", "acme-aud"))
print("acme->globex (tenant):", authorize("acme", "globex", "acme-aud", "globex-aud"))
print("acme tok, globex aud :", authorize("acme", "acme", "acme-aud", "globex-aud"))

## 🎯 Senior lens

The tell of a junior platform is a worker with a fat service account “to keep it simple” — simple until an indirect injection (41-01) turns it into every customer's data at once. **Scoped / short-lived / per-user / per-run** tokens are more plumbing up front, but they convert a platform-ending breach into a one-user incident. That is an expensive, hard-to-reverse architecture decision — and it is **yours, not the model's**.

## 📋 Production security checklist (§41) — copyable

Walk every agent feature against this before it ships.

In [ ]:
PRODUCTION_SECURITY_CHECKLIST = [
    "Threat model: is the lethal trifecta broken (no single agent has private data +"
    " untrusted content + an egress channel)?",
    "OWASP review: design walked against the LLM Top 10, accepted risks written down.",
    "Untrusted content: provenance marked; injection screening on docs/web/tool results,"
    " not just user input.",
    "Output handling: model output validated/escaped, never executed or rendered raw;"
    " URLs stripped or proxied.",
    "Guardrails: input+output guards at the gateway, no bypass path; PII redacted both"
    " directions; rejection rates monitored.",
    "Least privilege: minimum tools per agent, minimally-scoped credential per tool,"
    " tools act as the requesting user; per-tenant isolation.",
    "Approvals: irreversible/high-value actions behind human confirmation; tool tiers"
    " written down.",
    "Sandboxing: code execution in disposable containers; no ambient secrets; egress"
    " locked down; resource + time limits.",
    "Limits: rate/spend/iteration caps per agent and tenant; a kill switch you have"
    " actually tested.",
    "Audit: every tool call + guardrail decision in an append-only log with tenant/user/"
    "agent attribution; retention + anomaly alerts.",
    "Secrets: none in prompts/system messages; in a manager (Ch 36), rotated, absent"
    " from logs/traces.",
    "Privacy: lawful basis understood; provider retention/training terms reviewed;"
    " deletion reaches logs, caches, AND vector stores; residency honored.",
    "Compliance posture: SOC 2 / HIPAA / GDPR controls + evidence accruing now, not the"
    " quarter someone asks.",
    "Drills: have you red-teamed your own agents with injection payloads -- and does"
    " anything above fail when you do?",
]
for i, item in enumerate(PRODUCTION_SECURITY_CHECKLIST, 1):
    print(f"[ ] {i:>2}. {item}")

### A short privacy / compliance map

- **GDPR** data-minimization = your **PII redaction**; *right-to-erasure* must reach logs, caches, **and** vector stores.
- **SOC 2** = your **audit logs** are the evidence the auditor asks for.
- **HIPAA** = no BAA, **no PHI in prompts**, full stop.
- **Residency** is frequently the real reason to **self-host** (Ch 39), not cost.

Don't memorize statutes — build the *capabilities* every regime asks for (know-your-data, minimize/redact, isolate tenants, log access, delete on request, control geography), and each new law becomes configuration.

## Recap

- Guardrails inspect content; this layer constrains **capability** (Excessive Agency) — the antidote is **least privilege**.
- **Tier tools by consequence**; only irreversible/high-value actions stop for a human.
- **Sandbox** code with default-deny egress — data that can't leave can't be exfiltrated; bound blast radius with caps, a kill switch, and an audit log.
- **OAuth2 token exchange** mints per-user, scoped, short-lived grants; the **background worker** carries the delegation in the *job*, not a fat credential.
- **Per-tenant isolation** is the independent backstop — delegation gets the user right, isolation guarantees the boundary.

## Exercises

1. Add a `code_interpreter` tool at a new tier and route it through the policy. **Predict** whether it needs human approval, then justify the tier.
2. Give the `Sandbox` a one-host allowlist (`api.internal.example`) and show the injected `evil.example` egress still denies. **Predict** both log lines.
3. Set a grant's `lifetime_seconds` to `0`, sleep a tick, and show `redeem` raises — demonstrating short-lived tokens expire.
4. Break tenant isolation on purpose (return `True` from `authorize` for mismatched audiences) and write the assertion that *would have caught it* in CI.

In [ ]:
# Exercise 1 -- your code here


In [ ]:
# Exercise 2 -- your code here


In [ ]:
# Exercise 3 -- your code here


In [ ]:
# Exercise 4 -- your code here


## Next

That **closes Part X**: you can serve models (Ch 39), afford them (Ch 40), and now **defend** them. Part XI puts the whole book together — designing complete AI systems at scale.

- **Capstone:** these rails build **`capstone-project/security/`** (tool tiers + scopes, sandbox policy, delegated-auth / token-exchange, audit table) and add the per-tenant limit layer to `capstone-project/llm/gateway.py`; the delegated-grant path advances `capstone-project/workers/` (Ch 31). Checkpoint `checkpoints/ch41-security-and-guardrails`.
- **Blueprint:** the permission/sandbox rails harden [`../../blueprints/llm-gateway/`](../../blueprints/llm-gateway/) (Ch 39).
- **Template:** per-agent scopes/sandbox defaults harden [`../../templates/fastapi-agent-service/`](../../templates/fastapi-agent-service/) (Ch 25).